# Day17 — Approval Audit Trail + Permission Control

## Goal

离线演示启动绑定的本地审批角色、最小权限能力和可验证的哈希链审计记录。该教程不调用网络、LLM、Unity 或 Git。

## Setup

所有运行时文件均写入临时目录，结束时自动清理。

### Key Assumptions

- 从仓库根目录或 `day17/` 目录执行；
- Python 3.10+；
- 不需要 API Key、Unity Editor、登录账号或生成代码仓库。

In [ ]:
from pathlib import Path
import sys
import tempfile

current_directory = Path.cwd().resolve()
repository_root = current_directory if (current_directory / 'tools').is_dir() else current_directory.parent
assert (repository_root / 'tools' / 'approval_policy.py').is_file(), '请从仓库根目录或 day17 目录运行'
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from memory.approval_audit import ApprovalAuditStore, project_fingerprint
from tools.approval_policy import ApprovalPolicy

temporary_directory = tempfile.TemporaryDirectory()
runtime_root = Path(temporary_directory.name)
generated_root = runtime_root / 'generated'
generated_root.mkdir()

## Steps

### 1. 检查启动身份与能力

身份来自服务端启动配置。浏览器只显示能力，不能切换角色或提升权限。

In [ ]:
policies = {
    role: ApprovalPolicy.from_environment({
        'APPROVAL_ACTOR_ID': f'local-{role}',
        'APPROVAL_ACTOR_ROLE': role,
    })
    for role in ('viewer', 'reviewer', 'approver', 'operator')
}
role_capabilities = {role: policy.context()['capabilities'] for role, policy in policies.items()}
assert 'approval.decide' in role_capabilities['approver']
assert 'approval.decide' not in role_capabilities['reviewer']
assert 'task.operate' in role_capabilities['operator']
assert ApprovalPolicy.from_environment({}).actor.role == 'viewer'
print(role_capabilities)

### 2. 写入并链接有界审计事件

审计只保存相对文件元数据和哈希。备注中的密钥样式文本会在进入哈希链前清理。

In [ ]:
audit_store = ApprovalAuditStore(
    runtime_root / 'approval_audit.jsonl',
    project_fingerprint(generated_root),
)
file_evidence = [{
    'file': 'SafeCounter.cs',
    'operation': 'create',
    'before_hash': '0' * 64,
    'after_hash': '1' * 64,
}]
actor = policies['approver'].actor
proposal_event = audit_store.append({
    'event_type': 'proposal_created',
    'thread_id': 'day17-demo',
    'bundle_id': 'bundle-demo',
    'source': 'coder',
    'actor_id': actor.actor_id,
    'role': actor.role,
    'files': file_evidence,
    'action': 'create',
    'result': 'pending',
    'note': '',
    'error_code': '',
}, idempotency_key='proposal:day17-demo:bundle-demo')
decision_event = audit_store.append({
    'event_type': 'decision_authorized',
    'thread_id': 'day17-demo',
    'bundle_id': 'bundle-demo',
    'source': 'coder',
    'actor_id': actor.actor_id,
    'role': actor.role,
    'files': file_evidence,
    'action': 'approve:batch',
    'result': 'authorized',
    'note': 'token=demo-secret',
    'error_code': '',
}, idempotency_key='decision:day17-demo:bundle-demo:local-approver')
assert decision_event['previous_hash'] == proposal_event['event_hash']
print({'sequence': decision_event['sequence'], 'linked': True})

## Checks

重新读取完整链并执行只读导出。任何序号、前序哈希、项目指纹或事件内容被修改时，验证都会失败关闭。

In [ ]:
exported = audit_store.export_verified()
assert exported['verified'] is True
assert len(exported['events']) == 2
assert exported['events'][1]['note'] == 'token=[REDACTED]'
assert audit_store.verify() is True
day17_summary = {
    'verified': exported['verified'],
    'event_count': len(exported['events']),
    'last_sequence': exported['events'][-1]['sequence'],
    'network_used': False,
}
temporary_directory.cleanup()
print(day17_summary)

## Next Steps

1. 在本地 `.env` 中配置可信操作者和独立运行时审计路径；
2. 运行 `python -m tools.environment_check`，确认身份能力和审计目录；
3. 在开放远程观察或多人操作前，另行加入真实认证与安全会话；Day17 的启动身份不是登录系统。